**Author:** Ashutosh Jayant  
**Project:** India State Competitiveness Index (ISCI)

# 07 - State Competitiveness Profiles

Research question: how does each state perform on the index?
Builds one data-only profile per ranked state. No interpretation or policy here; that
starts in Notebook 08.

## Setup

Load the Version 1.0 outputs (read only), make the state_profiles folder, and work out
the national average and the top state.

In [1]:
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src import scoring

profiles_dir = root / "version2" / "reports" / "state_profiles"
profiles_dir.mkdir(parents=True, exist_ok=True)

scores = pd.read_csv(root / "results" / "indicator_scores.csv", index_col=0)
dimensions = pd.read_csv(root / "results" / "dimension_scores.csv", index_col=0)
index = pd.read_csv(root / "results" / "competitiveness_index.csv", index_col=0)

ranked = index[index["Rank"].notna()].sort_values("Rank")
national_avg = scores.mean()
top_state = ranked.index[0]

print("ranked states:", len(ranked), "| top state:", top_state)
print("profiles folder:", profiles_dir)

ranked states: 33 | top state: Goa
profiles folder: D:\india-state-competitiveness-index\version2\reports\state_profiles


## Profile data

A function that gathers all the numbers for one state: its rank and scores, its dimension
scores, its strongest and weakest indicators, and how it compares with the national
average and the top state. Pure data, no interpretation.

In [2]:
def profile_data(state):
    s = scores.loc[state].dropna()
    diff_nat = (scores.loc[state] - national_avg).dropna()
    diff_top = (scores.loc[state] - scores.loc[top_state]).dropna()

    fc_cols = [c for c in scoring.FACTOR_CONDITIONS if c in s.index]
    rel_cols = [c for c in scoring.RELATED_SUPPORTING if c in s.index]

    return {
        "rank": int(index.loc[state, "Rank"]),
        "score": round(index.loc[state, "competitiveness_score"], 3),
        "available": int(index.loc[state, "Indicators_Available"]),
        "coverage": int(index.loc[state, "Coverage_Percent"]),
        "fc_score": round(dimensions.loc[state, "factor_conditions"], 3),
        "rel_score": round(dimensions.loc[state, "related_supporting"], 3),
        "fc": s[fc_cols].sort_values(ascending=False),
        "rel": s[rel_cols].sort_values(ascending=False),
        "top5": s.sort_values(ascending=False).head(5),
        "bottom5": s.sort_values().head(5),
      "above_avg": diff_nat[diff_nat > 0].sort_values(ascending=False).head(3),
        "below_avg": diff_nat[diff_nat < 0].sort_values().head(3),
        "vs_top": diff_top.sort_values().head(3),
    }

d = profile_data("Gujarat")
print("rank:", d["rank"], "| score:", d["score"], "| coverage:", d["coverage"], "%")
print("FC score:", d["fc_score"], "| Related score:", d["rel_score"])
print("\ntop 5:");  print(d["top5"].round(3).to_string())
print("\nbottom 5:"); print(d["bottom5"].round(3).to_string())
print("\nmost above national average:"); print(d["above_avg"].round(3).to_string())
print("\nmost below national average:"); print(d["below_avg"].round(3).to_string())
print("\nfurthest below the top state:"); print(d["vs_top"].round(3).to_string())

rank: 3 | score: 0.594 | coverage: 100 %
FC score: 0.46 | Related score: 0.829

top 5:
unemployment_rate      1.000
manufacturing_share    0.992
investment_rate        0.930
factory_density        0.890
td_losses              0.783

bottom 5:
road_density        0.085
per_capita_power    0.124
ger                 0.255
life_expectancy     0.451
msme_density        0.505

most above national average:
manufacturing_share    0.636
investment_rate        0.627
factory_density        0.542

most below national average:
ger               -0.216
life_expectancy   -0.122
road_density      -0.110

furthest below the top state:
ger            -0.622
msme_density   -0.278
road_density   -0.253


## Build one profile

A function that turns a state's data into a Markdown profile, using a fixed template.
The observations are data facts, and the questions are left open for the later notebooks.

In [3]:
import re

ind_rank = scores.rank(ascending=False, method="min")   # 1 = highest score on that indicator
ind_count = scores.notna().sum()

def slug(name):
    return re.sub(r"[^a-z0-9]+", "_", name.lower().replace("&", "and")).strip("_")

def fmt(series):
    return ", ".join(f"{k} ({v:.2f})" for k, v in series.items())

def national_standing(state):
    r = ind_rank.loc[state].dropna()
    top = [c for c in r.index if r[c] <= 5]
    bottom = [c for c in r.index if r[c] >= ind_count[c] - 4]
    return top, bottom

def render_card(state):
    d = profile_data(state)
    top_nat, bottom_nat = national_standing(state)
    above = int((scores.loc[state] > national_avg).sum())
    higher = "higher" if d["rel_score"] > d["fc_score"] else "lower"

    L = [f"# {state}\n",
         f"Overall rank: {d['rank']} of 33  ",
         f"Overall score: {d['score']:.3f}  ",
         f"Coverage: {d['available']} of 11 indicators ({d['coverage']}%)\n",
         "## Quick summary\n",
         f"{state} ranks {d['rank']} of 33 with an overall score of {d['score']:.3f}. "
         f"Its Related & Supporting Industries score ({d['rel_score']:.3f}) is {higher} "
         f"than its Factor Conditions score ({d['fc_score']:.3f}).\n",
         "## Factor Conditions\n",
         f"Dimension score: {d['fc_score']:.3f}\n",
         f"Strongest: {fmt(d['fc'].head(2))}  ",
         f"Weakest: {fmt(d['fc'].tail(2))}\n",
         "## Related & Supporting Industries\n",
         f"Dimension score: {d['rel_score']:.3f}\n",
         f"Strongest: {fmt(d['rel'].head(2))}  ",
         f"Weakest: {fmt(d['rel'].tail(2))}\n",
         "## Top 5 indicators\n"]
    L += [f"- {k}: {v:.2f}" for k, v in d["top5"].items()] + [""]
    L += ["## Bottom 5 indicators\n"]
    L += [f"- {k}: {v:.2f}" for k, v in d["bottom5"].items()] + [""]
    L += ["## Comparison with the national average\n",
          f"Most above average: {fmt(d['above_avg'])}  ",
          f"Most below average: {fmt(d['below_avg'])}\n",
          f"## Comparison with the top-ranked state ({top_state})\n"]
    L += ["This is the top-ranked state.\n"] if state == top_state else \
         [f"Furthest below {top_state}: {fmt(d['vs_top'])}\n"]
    L += ["## Data-based observations\n"]
    if top_nat:
        L.append(f"- Among the 5 highest-scoring states on: {', '.join(top_nat)}.")
    if bottom_nat:
        L.append(f"- Among the 5 lowest-scoring states on: {', '.join(bottom_nat)}.")
    L.append(f"- Scores above the national average on {above} of {d['available']} available indicators.")
    L += ["", "## Questions suggested by the data for later analysis\n",
          f"- Why does {state} score high on {d['top5'].index[0]} but low on {d['bottom5'].index[0]}?",
          f"- Why is its Related & Supporting score {higher} than its Factor Conditions score?"]
    if state != top_state:
        L.append(f"- What explains the gap with {top_state} on {d['vs_top'].index[0]}?")
    L += ["", "_These are open questions, examined in Notebook 08 and later._"]
    return "\n".join(L)

print(render_card("Gujarat"))

# Gujarat

Overall rank: 3 of 33  
Overall score: 0.594  
Coverage: 11 of 11 indicators (100%)

## Quick summary

Gujarat ranks 3 of 33 with an overall score of 0.594. Its Related & Supporting Industries score (0.829) is higher than its Factor Conditions score (0.460).

## Factor Conditions

Dimension score: 0.460

Strongest: unemployment_rate (1.00), td_losses (0.78)  
Weakest: per_capita_power (0.12), road_density (0.09)

## Related & Supporting Industries

Dimension score: 0.829

Strongest: manufacturing_share (0.99), investment_rate (0.93)  
Weakest: factory_density (0.89), msme_density (0.51)

## Top 5 indicators

- unemployment_rate: 1.00
- manufacturing_share: 0.99
- investment_rate: 0.93
- factory_density: 0.89
- td_losses: 0.78

## Bottom 5 indicators

- road_density: 0.09
- per_capita_power: 0.12
- ger: 0.25
- life_expectancy: 0.45
- msme_density: 0.51

## Comparison with the national average

Most above average: manufacturing_share (0.64), investment_rate (0.63), factory_den

## Write all profiles

Generate a Markdown profile for every ranked state and save it in the state_profiles
folder, one file per state.

In [4]:
for state in ranked.index:
    (profiles_dir / f"{slug(state)}.md").write_text(render_card(state), encoding="utf-8")

files = sorted(profiles_dir.glob("*.md"))
print("wrote", len(files), "profiles to", profiles_dir)
for f in files:
    print(" ", f.name)

wrote 33 profiles to D:\india-state-competitiveness-index\version2\reports\state_profiles
  andaman_and_nicobar_islands.md
  andhra_pradesh.md
  arunachal_pradesh.md
  assam.md
  bihar.md
  chandigarh.md
  chhattisgarh.md
  delhi.md
  goa.md
  gujarat.md
  haryana.md
  himachal_pradesh.md
  jammu_and_kashmir.md
  jharkhand.md
  karnataka.md
  kerala.md
  madhya_pradesh.md
  maharashtra.md
  manipur.md
  meghalaya.md
  mizoram.md
  nagaland.md
  odisha.md
  puducherry.md
  punjab.md
  rajasthan.md
  sikkim.md
  tamil_nadu.md
  telangana.md
  tripura.md
  uttar_pradesh.md
  uttarakhand.md
  west_bengal.md
